# E-Commerce Cohort Retention Analysis

**Objective:** Analyze weekly user cohorts from an online retail dataset to identify drop-off points.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style='whitegrid', palette='rocket')

In [2]:
# Loading a synthetic, but realistic looking E-Commerce dataset
# For portfolio purposes, simulating 10k rows of transactions
np.random.seed(42)
dates = pd.date_range(start='2023-01-01', periods=120)
user_ids = np.random.randint(1000, 3000, size=10000)
login_dates = np.random.choice(dates, size=10000)

df = pd.DataFrame({'CustomerID': user_ids, 'InvoiceDate': login_dates})
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

In [3]:
# Get first purchase date for each user
first_logins = df.groupby('CustomerID')['InvoiceDate'].min().reset_index()
first_logins.columns = ['CustomerID', 'CohortDate']

df = pd.merge(df, first_logins, on='CustomerID')

# Convert to weekly periods
df['CohortWeek'] = df['CohortDate'].dt.to_period('W')
df['InvoiceWeek'] = df['InvoiceDate'].dt.to_period('W')

df['WeekNumber'] = (df['InvoiceWeek'] - df['CohortWeek']).apply(lambda x: x.n)
df.head()

In [4]:
cohort_data = df.groupby(['CohortWeek', 'WeekNumber'])['CustomerID'].nunique().reset_index()
retention_pivot = cohort_data.pivot(index='CohortWeek', columns='WeekNumber', values='CustomerID')
cohort_sizes = retention_pivot.iloc[:, 0]
retention_rates = retention_pivot.divide(cohort_sizes, axis=0)
retention_rates.round(3) * 100

In [5]:
plt.figure(figsize=(12, 8))
sns.heatmap(retention_rates, annot=False, cmap='Reds', vmin=0.0, vmax=0.5)
plt.title('Weekly Customer Retention Cohorts')
plt.ylabel('Cohort Week')
plt.xlabel('Weeks Since First Purchase')
plt.show()